# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

# Exercise

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

## Intro and purpose
This application is mean to provide a voice-to-voice interaction where users ask a question about a specific topic (in voice) and receive back an answere (written and audio)

## Setup

In [ ]:
area_of_expertise = "Ancien Rome History" # Please specify your topic

In [ ]:
system_message = f"""
You are an helpeful assistant expert in {area_of_expertise}.
You provide detailed answers to user questions on the topic.
Your answers are always structured as: a short, to-the-point explanation (one-line) and then an "analogy paragraph" that explains the concept using the rethorical figure of the analogy. This is crucial to make people understand your message

If the question is not about {area_of_expertise}, then you explicitly say you cannot answer the question as that is not your area of expertise.
"""

## Implementation

### Audio

In [ ]:
def tts(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message,
      speed=1.1
    )
    
    return response.content

In [ ]:
def asr(audio_file_path):

    if audio_file_path is None:
        return ""

    with open(audio_file_path, "rb") as audio_track: 
        response = openai.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe-2025-12-15",
            file=audio_track
        )

    return response.text

### Final UI

In [ ]:
def put_message_in_chatbot(recording, history):
        message = asr(recording)
        
        return history + [{"role":"user", "content":message}], None

In [ ]:
def audio_chat(history):

    history = [{"role": h["role"], "content":h['content']} for h in history]
    messages = [{"role" : "system", "content" : system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages)

    reply = response.choices[0].message.content
    history += [{"role" : "assistant", "content": reply}]
    voice = tts(reply)
    return history, voice

In [ ]:
# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
    with gr.Row():
        audio_input = gr.Microphone(label="Record a message here", sources=["microphone"], type="filepath")

    with gr.Row():
        audio_output = gr.Audio(label="Listen to the answer", autoplay=True)


# Hooking up events to callbacks
    audio_input.stop_recording(put_message_in_chatbot, inputs=[audio_input, chatbot], outputs=[chatbot, audio_input]).then(
        audio_chat, inputs=chatbot, outputs=[chatbot, audio_output]
    )



ui.launch(inbrowser=True)

In [ ]:
message = """The PROCEDURE DIVISION is the section of a COBOL program where the executable statements (the actual instructions) are written.

Think of a COBOL program like a recipe book: while earlier sections list the ingredients and tools needed (data definitions), the PROCEDURE DIVISION is like the step-by-step instructions the cook follows to prepare the meal — it's where you tell the computer exactly what actions to perform, in what order, to get the desired result."""

In [ ]:
audio = tts(message)

In [ ]:
from IPython.display import Audio, display

In [ ]:
display(Audio(audio, autoplay=True))